# Notebook 07 — Clone × Process Optimization Simulation

## Goal

Notebook07 moves from clone-only selection to clone-process pair selection.

Earlier notebooks asked:

> Which clone should we select?

Notebook07 asks:

> Which clone performs best under which process condition?

---

## Core idea

A clone's performance is not fixed.

Its final productivity, stability, and quality may depend on:

- media richness
- nutrient limitation
- feeding strategy
- metabolic stress
- aggregation risk

This notebook simulates conservative process-condition effects and evaluates whether process-aware clone-process pairing can improve selection.

---

## Strategy

We define several virtual process conditions:

- baseline
- rich_media
- nutrient_limited
- balanced_feed

For each clone × process pair, we estimate:

- process-adjusted late qP
- process-adjusted qP drop
- process-adjusted late aggregation
- process-adjusted utility

Then we select the best clone-process pairs.

In [1]:
# --------------------------------------------------
# Load prediction table from Notebook03
# --------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"   # "legacy" or "optimized"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / (
    f"predictions_03b_qp_{n_clones}_{scenario}.csv"
)

print("Loading:", PRED_PATH)

df = pd.read_csv(PRED_PATH)

print("Shape:", df.shape)
display(df.head())

required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)

assert len(missing) == 0, f"Missing required columns: {missing}"

Loading: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 19)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.498165,2.673129e-08,8.435996,0.238692,0,0.206868,3.215605e-08,4.404966,1,0.001494,0.003219,0.000312,0.506117,0.981713,0.996769,0.710487,0.568590,0
1,CLONE_2587,0.373417,8.407121e-08,3.130765,0.464397,0,0.600740,5.393417e-08,3.662670,0,0.007656,0.000000,0.013124,0.921942,0.000000,1.000000,0.435364,0.416664,0
2,CLONE_2654,0.461606,2.770586e-08,7.582349,0.256485,0,0.156409,3.459109e-08,10.144181,1,0.001523,0.004162,0.000529,0.627981,0.737814,0.995823,0.688798,0.538817,0
3,CLONE_1056,0.348761,7.746555e-08,6.885601,0.507813,1,0.465880,6.392818e-08,2.383546,0,0.012355,0.000000,0.011648,0.991742,0.538743,1.000000,0.382442,0.594558,0
4,CLONE_0706,0.644735,1.744327e-07,8.275030,0.018172,0,0.678633,1.526521e-07,5.032915,0,0.004539,0.039329,0.033316,0.017551,0.935723,0.960529,0.979289,0.407241,0


Missing required columns: []


## Step 1 — Utility helpers

We define utility scores for evaluating clone quality.

Higher utility means:

- higher late qP
- lower qP drop
- lower aggregation

The true utility is used only for offline evaluation.

In [2]:
# --------------------------------------------------
# Step 1 — Utility helper functions
# --------------------------------------------------

def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def z01(s):
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

def make_true_utility(df):
    return (
        1.0 * z(df["true_late_qp"])
        - 1.0 * z(df["true_qp_drop"])
        - 0.2 * z(df["true_late_agg"])
    )

def make_pred_utility(df, qp_col, drop_col, agg_col):
    return (
        1.0 * z(df[qp_col])
        - 1.0 * z(df[drop_col])
        - 0.2 * z(df[agg_col])
    )

def topk_overlap_by_id(true_ids, selected_ids, k=10):
    return len(set(true_ids) & set(selected_ids)) / k

df["true_util"] = make_true_utility(df)
df["baseline_score"] = make_pred_utility(
    df,
    qp_col="pred_late_qp",
    drop_col="pred_qp_drop",
    agg_col="pred_late_agg",
)

display(df[[
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_rescue_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
    "baseline_score",
]].head())

,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_util,baseline_score
0,CLONE_1502,2.673129e-08,0.498165,8.435996,0.568590,3.215605e-08,0.206868,4.404966,1.550478,-0.977650
1,CLONE_2587,8.407121e-08,0.373417,3.130765,0.416664,5.393417e-08,0.600740,3.662670,-0.804406,0.773624
2,CLONE_2654,2.770586e-08,0.461606,7.582349,0.538817,3.459109e-08,0.156409,10.144181,1.502466,-0.568919
3,CLONE_1056,7.746555e-08,0.348761,6.885601,0.594558,6.392818e-08,0.465880,2.383546,0.098570,0.649212
4,CLONE_0706,1.744327e-07,0.644735,8.275030,0.407241,1.526521e-07,0.678633,5.032915,-1.355847,-1.890491


## Step 2 — Define virtual process conditions

We define four conservative virtual process conditions.

These are not real bioreactor recipes.  
They are simplified process scenarios used to test clone × process interactions.

### Process conditions

- baseline: no change
- rich_media: modest qP gain, but possible stability / aggregation burden
- nutrient_limited: lower productivity, but improved stability and quality
- balanced_feed: small improvement across productivity, stability, and quality

The effect sizes are intentionally conservative.

In [3]:
# --------------------------------------------------
# Step 2 — Define conservative process conditions
# --------------------------------------------------
# Values represent baseline process effect before clone-specific sensitivity.
#
# qp_effect:
#   relative change in predicted late qP
#
# drop_effect:
#   absolute change in predicted qP drop
#   negative = improved stability
#   positive = worse stability
#
# agg_effect:
#   relative change in predicted aggregation
#   negative = reduced aggregation
#   positive = increased aggregation

process_conditions = {
    "baseline": {
        "qp_effect": 0.00,
        "drop_effect": 0.00,
        "agg_effect": 0.00,
    },
    "rich_media": {
        "qp_effect": 0.08,
        "drop_effect": 0.03,
        "agg_effect": 0.06,
    },
    "nutrient_limited": {
        "qp_effect": -0.04,
        "drop_effect": -0.08,
        "agg_effect": -0.08,
    },
    "balanced_feed": {
        "qp_effect": 0.04,
        "drop_effect": -0.04,
        "agg_effect": -0.04,
    },
}

pd.DataFrame(process_conditions).T

,qp_effect,drop_effect,agg_effect
baseline,0.00,0.00,0.00
rich_media,0.08,0.03,0.06
nutrient_limited,-0.04,-0.08,-0.08
balanced_feed,0.04,-0.04,-0.04


## Step 3 — Clone-specific process sensitivity

Different clones should respond differently to process changes.

We define conservative sensitivity proxies using existing predictions:

- rescue sensitivity:
  - higher rescue score means more process optimization potential

- instability sensitivity:
  - moderate qP drop may indicate room for stabilization
  - extreme instability should not be over-rewarded

- aggregation sensitivity:
  - moderate aggregation may be reducible by process tuning
  - extreme aggregation remains risky

This is still rule-based, not learned from real process data.

In [4]:
# --------------------------------------------------
# Step 3 — Clone-specific process sensitivity
# --------------------------------------------------

def triangular_band_score(x, low, high, peak=None):
    x = pd.Series(x).astype(float)
    if peak is None:
        peak = (low + high) / 2

    score = pd.Series(np.zeros(len(x)), index=x.index)

    left = (x >= low) & (x <= peak)
    right = (x > peak) & (x <= high)

    score.loc[left] = (x.loc[left] - low) / (peak - low + 1e-12)
    score.loc[right] = (high - x.loc[right]) / (high - peak + 1e-12)

    return score.clip(0, 1)

work = df.copy()

work["sens_rescue"] = z01(work["pred_rescue_score"])

work["sens_stability_band"] = triangular_band_score(
    work["pred_qp_drop"],
    low=0.20,
    peak=0.35,
    high=0.60,
)

work["sens_agg_band"] = triangular_band_score(
    work["pred_late_agg"],
    low=5.0,
    peak=9.0,
    high=16.0,
)

work["process_sensitivity"] = (
    0.55 * work["sens_rescue"]
    + 0.25 * work["sens_stability_band"]
    + 0.20 * work["sens_agg_band"]
)

print("=== Process sensitivity summary ===")
display(work["process_sensitivity"].describe())

display(work[[
    "clone_id",
    "pred_rescue_score",
    "pred_qp_drop",
    "pred_late_agg",
    "sens_rescue",
    "sens_stability_band",
    "sens_agg_band",
    "process_sensitivity",
]].sort_values("process_sensitivity", ascending=False).head(15))

=== Process sensitivity summary ===


count    1000.000000
mean        0.469484
std         0.166124
min         0.000000
25%         0.362534
50%         0.454547
75%         0.586342
max         0.938486
Name: process_sensitivity, dtype: float64

,clone_id,pred_rescue_score,pred_qp_drop,pred_late_agg,sens_rescue,sens_stability_band,sens_agg_band,process_sensitivity
180,CLONE_4625,1.000000,0.322967,8.670837,1.000000,0.819778,0.917709,0.938486
269,CLONE_0150,0.714357,0.353381,8.895115,0.714357,0.986475,0.973779,0.834271
690,CLONE_4335,0.717382,0.345661,8.935721,0.717382,0.971074,0.983930,0.834114
825,CLONE_1490,0.725636,0.351665,8.633797,0.725636,0.993340,0.908449,0.829125
543,CLONE_4535,0.703903,0.361335,9.033933,0.703903,0.954658,0.995152,0.824841
182,CLONE_1942,0.733136,0.353795,8.453091,0.733136,0.984822,0.863273,0.822085
569,CLONE_3867,0.697816,0.364354,9.033906,0.697816,0.942583,0.995156,0.818476
967,CLONE_1421,0.690321,0.346262,9.278052,0.690321,0.975083,0.960278,0.815503
743,CLONE_4054,0.688747,0.368455,8.998400,0.688747,0.926180,0.999600,0.810276
716,CLONE_3693,0.706720,0.362765,8.547862,0.706720,0.948940,0.886966,0.803324


## Step 4 — Build clone × process table

We expand the dataset so that each clone appears once per process condition.

This allows us to evaluate:

> clone A under baseline  
> clone A under rich media  
> clone A under nutrient limitation  
> clone A under balanced feed

The selection problem becomes:

> Which clone-process pair is best?

In [5]:
# --------------------------------------------------
# Step 4 — Expand clones into clone × process pairs
# --------------------------------------------------

rows = []

for process_name, effect in process_conditions.items():
    tmp = work.copy()
    tmp["process_condition"] = process_name
    tmp["base_qp_effect"] = effect["qp_effect"]
    tmp["base_drop_effect"] = effect["drop_effect"]
    tmp["base_agg_effect"] = effect["agg_effect"]
    rows.append(tmp)

pair_df = pd.concat(rows, ignore_index=True)

print("Clone-process table shape:", pair_df.shape)
print("Number of unique clones:", pair_df["clone_id"].nunique())
print("Process conditions:", pair_df["process_condition"].unique())

display(pair_df[[
    "clone_id",
    "process_condition",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "process_sensitivity",
]].head(12))

Clone-process table shape: (4000, 29)
Number of unique clones: 1000
Process conditions: <StringArray>
['baseline', 'rich_media', 'nutrient_limited', 'balanced_feed']
Length: 4, dtype: str


,clone_id,process_condition,pred_late_qp,pred_qp_drop,pred_late_agg,process_sensitivity
0,CLONE_1502,baseline,2.673129e-08,0.498165,8.435996,0.586360
1,CLONE_2587,baseline,8.407121e-08,0.373417,3.130765,0.455748
2,CLONE_2654,baseline,2.770586e-08,0.461606,7.582349,0.563861
3,CLONE_1056,baseline,7.746555e-08,0.348761,6.885601,0.669223
4,CLONE_0706,baseline,1.744327e-07,0.644735,8.275030,0.387734
5,CLONE_0107,baseline,7.282213e-08,0.564464,5.454678,0.201960
6,CLONE_0590,baseline,3.186999e-07,0.545583,6.090235,0.304415
7,CLONE_2469,baseline,3.510977e-08,0.437753,4.023583,0.355122
8,CLONE_2414,baseline,4.016393e-08,0.391796,4.754340,0.428630
9,CLONE_1601,baseline,1.447429e-07,0.374386,3.500999,0.463722


## Step 5 — Apply clone-specific process effects

Each process condition has a base effect.

The actual effect is modulated by clone-specific sensitivity.

This creates clone × process interaction:

- rich media may help high-rescue clones more
- nutrient limitation may help unstable or aggregation-prone clones
- balanced feed may provide moderate all-around improvement

The effects remain conservative to avoid unrealistic ranking flips.

In [6]:
# --------------------------------------------------
# Step 5 — Apply clone-specific process effects
# --------------------------------------------------

# Clone-specific modulation
# Kept conservative to prevent artificial over-promotion.
pair_df["response_multiplier"] = 0.50 + 0.50 * pair_df["process_sensitivity"]

# Rich media can increase productivity more in process-sensitive clones,
# but can also increase aggregation burden.
is_rich = pair_df["process_condition"] == "rich_media"

# Nutrient limitation helps stability / aggregation more for sensitive clones,
# but may reduce qP slightly.
is_limited = pair_df["process_condition"] == "nutrient_limited"

# Balanced feed gives smaller but safer improvements.
is_balanced = pair_df["process_condition"] == "balanced_feed"

# Default adjusted effects
pair_df["adj_qp_effect"] = pair_df["base_qp_effect"] * pair_df["response_multiplier"]
pair_df["adj_drop_effect"] = pair_df["base_drop_effect"] * pair_df["response_multiplier"]
pair_df["adj_agg_effect"] = pair_df["base_agg_effect"] * pair_df["response_multiplier"]

# Extra conservative risk adjustment:
# rich media aggregation penalty is stronger for high predicted aggregation clones
pair_df.loc[is_rich, "adj_agg_effect"] = (
    pair_df.loc[is_rich, "adj_agg_effect"]
    * (1.0 + 0.30 * z01(pair_df.loc[is_rich, "pred_late_agg"]))
)

# Nutrient limitation stability benefit is stronger for moderate instability
pair_df.loc[is_limited, "adj_drop_effect"] = (
    pair_df.loc[is_limited, "adj_drop_effect"]
    * (1.0 + 0.30 * pair_df.loc[is_limited, "sens_stability_band"])
)

# Balanced feed quality benefit is stronger for moderate aggregation
pair_df.loc[is_balanced, "adj_agg_effect"] = (
    pair_df.loc[is_balanced, "adj_agg_effect"]
    * (1.0 + 0.20 * pair_df.loc[is_balanced, "sens_agg_band"])
)

# Apply effects
pair_df["process_late_qp"] = (
    pair_df["pred_late_qp"] * (1.0 + pair_df["adj_qp_effect"])
)

pair_df["process_qp_drop"] = (
    pair_df["pred_qp_drop"] + pair_df["adj_drop_effect"]
).clip(lower=0.0)

pair_df["process_late_agg"] = (
    pair_df["pred_late_agg"] * (1.0 + pair_df["adj_agg_effect"])
).clip(lower=0.0)

display(pair_df[[
    "clone_id",
    "process_condition",
    "process_sensitivity",
    "adj_qp_effect",
    "adj_drop_effect",
    "adj_agg_effect",
    "pred_late_qp",
    "process_late_qp",
    "pred_qp_drop",
    "process_qp_drop",
    "pred_late_agg",
    "process_late_agg",
]].sort_values("process_sensitivity", ascending=False).head(20))

,clone_id,process_condition,process_sensitivity,adj_qp_effect,adj_drop_effect,adj_agg_effect,pred_late_qp,process_late_qp,pred_qp_drop,process_qp_drop,pred_late_agg,process_late_agg
2180,CLONE_4625,nutrient_limited,0.938486,-0.038770,-0.096609,-0.077539,4.500575e-06,4.326089e-06,0.322967,0.226358,8.670837,7.998505
180,CLONE_4625,baseline,0.938486,0.000000,0.000000,0.000000,4.500575e-06,4.500575e-06,0.322967,0.322967,8.670837,8.670837
1180,CLONE_4625,rich_media,0.938486,0.077539,0.029077,0.069037,4.500575e-06,4.849547e-06,0.322967,0.352044,8.670837,9.269444
3180,CLONE_4625,balanced_feed,0.938486,0.038770,-0.038770,-0.045886,4.500575e-06,4.675061e-06,0.322967,0.284197,8.670837,8.272970
3269,CLONE_0150,balanced_feed,0.834271,0.036685,-0.036685,-0.043830,8.947176e-08,9.275407e-08,0.353381,0.316696,8.895115,8.505241
2269,CLONE_0150,nutrient_limited,0.834271,-0.036685,-0.095084,-0.073371,8.947176e-08,8.618945e-08,0.353381,0.258297,8.895115,8.242473
1269,CLONE_0150,rich_media,0.834271,0.073371,0.027514,0.065669,8.947176e-08,9.603638e-08,0.353381,0.380895,8.895115,9.479244
269,CLONE_0150,baseline,0.834271,0.000000,0.000000,0.000000,8.947176e-08,8.947176e-08,0.353381,0.353381,8.895115,8.895115
3690,CLONE_4335,balanced_feed,0.834114,0.036682,-0.036682,-0.043901,3.366004e-07,3.489477e-07,0.345661,0.308979,8.935721,8.543435
690,CLONE_4335,baseline,0.834114,0.000000,0.000000,0.000000,3.366004e-07,3.366004e-07,0.345661,0.345661,8.935721,8.935721


## Step 6 — Compute clone-process utility

Now each clone-process pair receives a utility score.

The score uses process-adjusted predicted values:

- process_late_qp
- process_qp_drop
- process_late_agg

This score represents expected performance under a specific process condition.

In [7]:
# --------------------------------------------------
# Step 6 — Compute clone-process utility
# --------------------------------------------------

pair_df["process_pair_score"] = make_pred_utility(
    pair_df,
    qp_col="process_late_qp",
    drop_col="process_qp_drop",
    agg_col="process_late_agg",
)

display(pair_df[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_util",
]].sort_values("process_pair_score", ascending=False).head(15))

,clone_id,process_condition,process_pair_score,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_util
3621,CLONE_3254,balanced_feed,13.194688,0.000004,0.212682,8.028645,0.787107,1,2.451734
3180,CLONE_4625,balanced_feed,13.104491,0.000005,0.284197,8.272970,1.000000,1,3.040391
1621,CLONE_3254,rich_media,13.012084,0.000005,0.271404,8.851701,0.787107,1,2.451734
1180,CLONE_4625,rich_media,12.907302,0.000005,0.352044,9.269444,1.000000,1,3.040391
2621,CLONE_3254,nutrient_limited,12.777295,0.000004,0.172920,7.795304,0.787107,1,2.451734
2180,CLONE_4625,nutrient_limited,12.687865,0.000004,0.226358,7.998505,1.000000,1,3.040391
621,CLONE_3254,baseline,12.494169,0.000004,0.246238,8.356089,0.787107,1,2.451734
180,CLONE_4625,baseline,12.275594,0.000005,0.322967,8.670837,1.000000,1,3.040391
3505,CLONE_0048,balanced_feed,9.653807,0.000004,0.533175,10.299777,0.701157,1,1.893128
1505,CLONE_0048,rich_media,9.442908,0.000004,0.588223,11.313866,0.701157,1,1.893128


## Step 7 — Select best process condition per clone

For each clone, we select the process condition with the highest process-pair score.

This converts the problem from:

> clone × process pair ranking

back to:

> one best expected process condition per clone

In [8]:
# --------------------------------------------------
# Step 7 — Best process per clone
# --------------------------------------------------

best_pair_per_clone = (
    pair_df.sort_values("process_pair_score", ascending=False)
    .groupby("clone_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best-pair table shape:", best_pair_per_clone.shape)

print("\n=== Best process condition distribution ===")
display(best_pair_per_clone["process_condition"].value_counts())

display(best_pair_per_clone[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].sort_values("process_pair_score", ascending=False).head(15))

Best-pair table shape: (1000, 37)

=== Best process condition distribution ===


process_condition
nutrient_limited    992
balanced_feed         8
Name: count, dtype: int64

,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_3254,balanced_feed,13.194688,12.974673,0.677776,0.787107,1,4.473668e-06,0.212682,8.028645,7.103148e-06,0.089331,11.556387,2.451734
1,CLONE_4625,balanced_feed,13.104491,12.718873,0.938486,1.000000,1,4.675061e-06,0.284197,8.272970,1.238325e-05,0.084353,10.114743,3.040391
2,CLONE_0048,balanced_feed,9.653807,9.293532,0.572806,0.701157,1,4.239081e-06,0.533175,10.299777,1.645542e-05,0.304950,12.743312,1.893128
3,CLONE_1619,balanced_feed,8.888315,8.901264,0.257379,0.358295,0,2.696743e-06,0.211043,3.272370,2.544839e-06,0.164530,5.259631,1.978856
4,CLONE_2171,balanced_feed,8.400434,7.964768,0.564704,0.711942,1,4.171479e-06,0.682191,8.152032,6.600422e-06,0.839031,6.662415,-1.862933
5,CLONE_2776,balanced_feed,8.308421,7.992076,0.415680,0.577185,0,3.856750e-06,0.608567,6.748002,3.556629e-04,0.000000,8.477892,34.138263
6,CLONE_4878,balanced_feed,8.281448,7.933686,0.638122,0.731989,1,3.626379e-06,0.512785,9.288479,4.912223e-06,0.417981,7.149145,0.525635
7,CLONE_3863,balanced_feed,5.477714,5.105205,0.460688,0.581613,0,3.141937e-06,0.709254,7.555532,2.084186e-06,0.801780,5.469702,-1.962704
8,CLONE_0643,nutrient_limited,4.722220,4.466508,0.594597,0.650158,1,1.885079e-06,0.409393,6.913095,1.701110e-06,0.518817,6.462675,-0.332074
9,CLONE_0101,nutrient_limited,3.508672,3.104626,0.719585,0.712995,1,1.173833e-06,0.325554,7.216114,1.143915e-06,0.514718,5.009296,-0.266375


## Step 8 — Compare baseline clone selection vs clone-process selection

We compare:

1. baseline Top10 selected by original predicted utility
2. clone-process Top10 selected by best process-pair score

This tells us whether process pairing changes clone selection.

In [9]:
# --------------------------------------------------
# Step 8 — Top10 comparison
# --------------------------------------------------

TOP_K = 10

baseline_top10 = (
    work.sort_values("baseline_score", ascending=False)
    .head(TOP_K)
    .copy()
)

process_pair_top10 = (
    best_pair_per_clone.sort_values("process_pair_score", ascending=False)
    .head(TOP_K)
    .copy()
)

baseline_ids = set(baseline_top10["clone_id"])
process_ids = set(process_pair_top10["clone_id"])

print("=== Top10 comparison ===")
print("Baseline rescue count:", int(baseline_top10["pred_rescue_label"].sum()))
print("Process-pair rescue count:", int(process_pair_top10["pred_rescue_label"].sum()))
print(f"Top10 overlap: {len(baseline_ids & process_ids)}/{TOP_K}")

print("\n=== Baseline Top10 ===")
display(baseline_top10[[
    "clone_id",
    "baseline_score",
    "pred_rescue_score",
    "pred_rescue_label",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

print("\n=== Process-pair Top10 ===")
display(process_pair_top10[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Top10 comparison ===
Baseline rescue count: 6
Process-pair rescue count: 7
Top10 overlap: 9/10

=== Baseline Top10 ===


,clone_id,baseline_score,pred_rescue_score,pred_rescue_label,pred_late_qp,pred_qp_drop,pred_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
621,CLONE_3254,12.974673,0.787107,1,4.328425e-06,0.246238,8.356089,0.000007,0.089331,11.556387,2.451734
180,CLONE_4625,12.718873,1.000000,1,4.500575e-06,0.322967,8.670837,0.000012,0.084353,10.114743,3.040391
505,CLONE_0048,9.293532,0.701157,1,4.109802e-06,0.564631,10.686979,0.000016,0.304950,12.743312,1.893128
314,CLONE_1619,8.901264,0.358295,0,2.630590e-06,0.236190,3.356785,0.000003,0.164530,5.259631,1.978856
986,CLONE_2776,7.992076,0.577185,0,3.750558e-06,0.636881,6.964564,0.000356,0.000000,8.477892,34.138263
524,CLONE_2171,7.964768,0.711942,1,4.044897e-06,0.713485,8.462717,0.000007,0.839031,6.662415,-1.862933
730,CLONE_4878,7.933686,0.731989,1,3.511339e-06,0.545547,9.662363,0.000005,0.417981,7.149145,0.525635
63,CLONE_3863,5.105205,0.581613,0,3.052754e-06,0.738468,7.816017,0.000002,0.801780,5.469702,-1.962704
23,CLONE_0643,4.466508,0.650158,1,1.947179e-06,0.482194,7.384080,0.000002,0.518817,6.462675,-0.332074
476,CLONE_2759,3.183487,0.481011,0,8.872743e-07,0.273353,10.806613,0.000001,0.220844,16.093701,0.834379



=== Process-pair Top10 ===


,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_3254,balanced_feed,13.194688,12.974673,0.677776,0.787107,1,0.000004,0.212682,8.028645,0.000007,0.089331,11.556387,2.451734
1,CLONE_4625,balanced_feed,13.104491,12.718873,0.938486,1.000000,1,0.000005,0.284197,8.272970,0.000012,0.084353,10.114743,3.040391
2,CLONE_0048,balanced_feed,9.653807,9.293532,0.572806,0.701157,1,0.000004,0.533175,10.299777,0.000016,0.304950,12.743312,1.893128
3,CLONE_1619,balanced_feed,8.888315,8.901264,0.257379,0.358295,0,0.000003,0.211043,3.272370,0.000003,0.164530,5.259631,1.978856
4,CLONE_2171,balanced_feed,8.400434,7.964768,0.564704,0.711942,1,0.000004,0.682191,8.152032,0.000007,0.839031,6.662415,-1.862933
5,CLONE_2776,balanced_feed,8.308421,7.992076,0.415680,0.577185,0,0.000004,0.608567,6.748002,0.000356,0.000000,8.477892,34.138263
6,CLONE_4878,balanced_feed,8.281448,7.933686,0.638122,0.731989,1,0.000004,0.512785,9.288479,0.000005,0.417981,7.149145,0.525635
7,CLONE_3863,balanced_feed,5.477714,5.105205,0.460688,0.581613,0,0.000003,0.709254,7.555532,0.000002,0.801780,5.469702,-1.962704
8,CLONE_0643,nutrient_limited,4.722220,4.466508,0.594597,0.650158,1,0.000002,0.409393,6.913095,0.000002,0.518817,6.462675,-0.332074
9,CLONE_0101,nutrient_limited,3.508672,3.104626,0.719585,0.712995,1,0.000001,0.325554,7.216114,0.000001,0.514718,5.009296,-0.266375


## Step 9 — Utility overlap and true performance

We evaluate whether clone-process selection better recovers truly good clones.

Utility overlap compares:

- true Top10 by true_util
- baseline selected Top10
- process-pair selected Top10

We also compare true performance averages.

In [10]:
# --------------------------------------------------
# Step 9 — Utility overlap and true performance
# --------------------------------------------------

true_top10 = work.sort_values("true_util", ascending=False).head(TOP_K)
true_top_ids = set(true_top10["clone_id"])

baseline_overlap = len(true_top_ids & set(baseline_top10["clone_id"])) / TOP_K
process_pair_overlap = len(true_top_ids & set(process_pair_top10["clone_id"])) / TOP_K

print("=== Utility overlap ===")
print(f"Baseline utility overlap:     {baseline_overlap:.3f}")
print(f"Process-pair utility overlap: {process_pair_overlap:.3f}")

summary = pd.DataFrame({
    "baseline_top10": baseline_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
    "process_pair_top10": process_pair_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
})

summary["direction"] = [
    "higher is better",
    "lower is better",
    "lower is better",
    "higher is better",
]

display(summary)

=== Utility overlap ===
Baseline utility overlap:     0.300
Process-pair utility overlap: 0.300


,baseline_top10,process_pair_top10,direction
true_late_qp,0.000041,0.000041,higher is better
true_qp_drop,0.344162,0.373549,lower is better
true_late_agg,8.998960,7.890520,lower is better
true_util,4.070468,3.960392,higher is better


## Step 10 — Process condition diagnostics

This step checks which process conditions are selected among top candidates.

This helps interpret whether the process model behaves reasonably.

Expected behavior:

- rich_media may appear for productivity-driven clones
- nutrient_limited may appear for unstable or aggregation-prone clones
- balanced_feed should often be a safe compromise

In [11]:
# --------------------------------------------------
# Step 10 — Process condition diagnostics
# --------------------------------------------------

print("=== Process condition distribution in Top10 ===")
display(process_pair_top10["process_condition"].value_counts())

print("\n=== Mean predicted process-adjusted outcomes by selected process ===")
display(process_pair_top10.groupby("process_condition")[[
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "process_pair_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].mean())

print("\n=== Rescue label distribution by selected process ===")
display(pd.crosstab(
    process_pair_top10["process_condition"],
    process_pair_top10["pred_rescue_label"],
    rownames=["process_condition"],
    colnames=["pred_rescue_label"],
))

=== Process condition distribution in Top10 ===


process_condition
balanced_feed       8
nutrient_limited    2
Name: count, dtype: int64


=== Mean predicted process-adjusted outcomes by selected process ===


,process_late_qp,process_qp_drop,process_late_agg,process_pair_score,true_late_qp,true_qp_drop,true_late_agg,true_util
process_condition,,,,,,,,
balanced_feed,0.000004,0.469237,7.702226,9.413665,0.000051,0.337744,8.429153,5.025296
nutrient_limited,0.000002,0.367473,7.064604,4.115446,0.000001,0.516768,5.735986,-0.299224



=== Rescue label distribution by selected process ===


pred_rescue_label,0,1
process_condition,,
balanced_feed,3,5
nutrient_limited,0,2


## Step 11 — Production Top3 guardrail

Top10 can include exploratory clone-process pairs.

However, Top3 should remain conservative because these represent production-like candidates.

Here we select Top3 from process-pair Top10 but require:

- acceptable predicted qP drop
- acceptable predicted aggregation

In [12]:
# --------------------------------------------------
# Step 11 — Conservative Top3 production candidates
# --------------------------------------------------

TOP3_MAX_DROP = 0.45
TOP3_MAX_AGG = 14.0

top3_pool = process_pair_top10[
    (process_pair_top10["process_qp_drop"] <= TOP3_MAX_DROP) &
    (process_pair_top10["process_late_agg"] <= TOP3_MAX_AGG)
].copy()

process_pair_top3 = (
    top3_pool.sort_values("process_pair_score", ascending=False)
    .head(3)
    .copy()
)

print("Top3 candidates:", len(process_pair_top3))

display(process_pair_top3[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

Top3 candidates: 3


,clone_id,process_condition,process_pair_score,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_3254,balanced_feed,13.194688,0.000004,0.212682,8.028645,0.787107,1,0.000007,0.089331,11.556387,2.451734
1,CLONE_4625,balanced_feed,13.104491,0.000005,0.284197,8.272970,1.000000,1,0.000012,0.084353,10.114743,3.040391
3,CLONE_1619,balanced_feed,8.888315,0.000003,0.211043,3.272370,0.358295,0,0.000003,0.164530,5.259631,1.978856


## Final Interpretation

Notebook07 introduces a clone × process selection framework.

Instead of selecting clones alone, this notebook evaluates:

> Which clone-process pair is expected to perform best?

---

## What this notebook demonstrates

1. Clone ranking can change under different virtual process conditions.
2. Process response is clone-specific rather than uniform.
3. Rescue clones may become more competitive under selected process conditions.
4. Top10 can be used as an exploratory clone-process candidate pool.
5. Top3 should remain protected by conservative production guardrails.

---

## Scientific basis

The simulation assumptions are intentionally conservative.

They are motivated by established CHO cell culture observations:

- Fed-batch feeding strategy and nutrient control influence cell growth, productivity, and product quality.
- Dynamic feeding strategies using real-time glucose monitoring have been applied in CHO culture for monoclonal antibody production.
- Media/feed combinations can affect titer and glycosylated antibody quality.
- Continuous or controlled feeding can reduce metabolic byproducts and improve antibody expression.
- Product quality attributes such as glycosylation are sensitive to culture conditions.

---

## References

1. Zalai et al. investigated using specific productivity as a physiological control parameter in mammalian cell culture processes.
2. Domján et al. developed Raman-based dynamic feeding strategies using real-time glucose monitoring in adalimumab-producing CHO cell culture.
3. Lu et al. demonstrated automated dynamic fed-batch process and media optimization for mammalian cell culture.
4. Xiao et al. showed that continuous feeding can reduce metabolic byproduct generation and improve antibody expression in CHO-K1 cultures.
5. Gyorgypal et al. evaluated the impact of media and feed combinations on glycosylated monoclonal antibody production using CHO cells.

---

## Current limitation

This notebook is still a simulation.

The process effects are not learned from real LC/GC, spent media, Raman, or bioreactor data.

Therefore, results should be interpreted as:

> A hypothesis-generating clone × process simulation framework

rather than a validated bioprocess model.

---

## Next step

If this notebook produces meaningful shifts in ranking, the next step is to formalize process-related features and later move them into:

- Notebook02 feature engineering
- src/simulation/process_effects.py
- src/decision/clone_process_selection.py